# 02 - Feature Engineering et Modélisation Baseline

**Projet** : Scoring de Crédit et Prévision de Défauts  
**Auteur** : Shafin Hamjah (Data Engineer)  
**Date** : Janvier 2026

---

## Objectifs de ce notebook

1. **Feature Engineering** : Créer de nouvelles variables prédictives
2. **Préparation des données** : Split temporel et encodage
3. **Modélisation Baseline** : Régression logistique avec SMOTE
4. **Évaluation** : Métriques et visualisations
5. **Analyse temporelle** : Étude préliminaire de la série des défauts

## 0. Configuration et imports

In [ ]:
# Imports standards
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve,
    classification_report, confusion_matrix, f1_score,
    precision_score, recall_score, accuracy_score
)
import joblib

# Configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

# Seed pour reproductibilité
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Chemins
ROOT_DIR = Path('..')
DATA_PATH = ROOT_DIR / 'data' / 'processed' / 'credit_data_cleaned.parquet'
FIGURES_PATH = ROOT_DIR / 'outputs' / 'figures' / '02_modeling'
MODELS_PATH = ROOT_DIR / 'outputs' / 'models'

# Créer les dossiers si nécessaire
FIGURES_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

print("Configuration terminée ✓")
print(f"Fichier de données : {DATA_PATH}")

## 1. Chargement des données nettoyées

In [ ]:
# Chargement des données prétraitées
df = pd.read_parquet(DATA_PATH)

print(f"Dimensions : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
print(f"Mémoire : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\nPériode : {df['DATE_MOIS'].min()} à {df['DATE_MOIS'].max()}")

In [ ]:
# Rappel de la distribution de la cible
target_dist = df['TARGET'].value_counts()
target_pct = df['TARGET'].value_counts(normalize=True) * 100

print("Distribution TARGET :")
print(f"  0 (Non défaut) : {target_dist[0]:,} ({target_pct[0]:.2f}%)")
print(f"  1 (Défaut)     : {target_dist[1]:,} ({target_pct[1]:.2f}%)")
print(f"\n⚠️ Ratio de déséquilibre : 1:{target_dist[0]/target_dist[1]:.1f}")

In [ ]:
# Aperçu des colonnes disponibles
print(f"Colonnes ({len(df.columns)}) :")
for i, col in enumerate(df.columns, 1):
    print(f"{i:2}. {col}", end="\t\t" if i % 3 != 0 else "\n")

---

## 2. Feature Engineering

Création de **6 variables ciblées**, chacune capturant un aspect distinct du risque de crédit :

| # | Variable | Dimension |
|---|----------|-----------|
| 1 | `CAPACITE_REMBOURSEMENT` | Capacité financière nette |
| 2 | `RATIO_PRET_REVENU_ANNUEL` | Proportionnalité de l'emprunt |
| 3 | `NB_TRANSACTIONS_TOTAL_30J` | Engagement comportemental |
| 4 | `TENDANCE_EPARGNE` | Trajectoire financière |
| 5 | `FLAG_HAUT_RISQUE` | Historique de risque |
| 6 | `ENDETTEMENT_X_UTILISATION` | Double contrainte de crédit |


### 2.1 Variables financières

In [ ]:
# Copie pour le feature engineering
df_fe = df.copy()
initial_cols = len(df_fe.columns)

# --- FEATURE 1 : CAPACITE_REMBOURSEMENT ---
# Capacité nette mensuelle de remboursement = revenu - dépenses - mensualité.
# Si négatif → le client ne peut pas couvrir sa mensualité avec ce qui lui reste.
# C'est la variable la plus directement liée à la défaillance.
df_fe['CAPACITE_REMBOURSEMENT'] = df_fe['REVENU_MENSUEL'] - df_fe['DEPENSES_MENSUELLES'] - df_fe['MENSUALITE']

# --- FEATURE 2 : RATIO_PRET_REVENU_ANNUEL ---
# Ratio montant du prêt / revenu annuel.
# Mesure si l'emprunt est proportionné aux moyens du client.
# Un ratio élevé (> 5x) signale un risque de sur-endettement.
df_fe['RATIO_PRET_REVENU_ANNUEL'] = df_fe['MONTANT_PRET'] / (df_fe['REVENU_MENSUEL'] * 12 + 1)

print("✓ CAPACITE_REMBOURSEMENT — capacité nette mensuelle après charges")
print("✓ RATIO_PRET_REVENU_ANNUEL — ratio prêt / revenu annuel")


### 2.2 Variable comportementale

In [ ]:
# --- FEATURE 3 : NB_TRANSACTIONS_TOTAL_30J ---
# Volume total de transactions (POS + ATM + en ligne) sur 30 jours.
# Un client peu actif est souvent moins engagé financièrement et présente un risque plus élevé.
# Proxy d'activité bancaire réelle, indépendant des niveaux de soldes.
df_fe['NB_TRANSACTIONS_TOTAL_30J'] = df_fe['NB_ACHATS_TPE_30J'] + df_fe['NB_RETRAITS_GAB_30J'] + df_fe['NB_ACHATS_EN_LIGNE_30J']

print("✓ NB_TRANSACTIONS_TOTAL_30J — activité transactionnelle totale sur 30 jours")


### 2.3 Variable de tendance

In [ ]:
# --- FEATURE 4 : TENDANCE_EPARGNE ---
# Tendance de l'épargne sur 3 mois : SAVINGS_M1 - SAVINGS_M3.
# Positif = épargne croissante (bonne santé financière).
# Négatif = le client consomme ses réserves — signal d'alerte prospectif.
# Capture une dynamique temporelle que les niveaux instantanés ne voient pas.
df_fe['TENDANCE_EPARGNE'] = df_fe['SOLDE_EPARGNE_M1'] - df_fe['SOLDE_EPARGNE_M3']

print("✓ TENDANCE_EPARGNE — tendance de l'épargne (M1 - M3)")


### 2.4 Variable de risque historique

In [ ]:
# --- FEATURE 5 : FLAG_HAUT_RISQUE ---
# Flag binaire combinant les signaux historiques les plus forts :
#   - >= 2 retards de paiement sur 12 mois
#   - Défaut passé (DEFAUT_ANTERIEUR)
#   - Credit score < 550 (zone très dégradée)
#   - DTI > 0.6 (60 % du revenu absorbé par les dettes)
# Synthèse efficace : un seul bit capture les combinaisons à haut risque.
df_fe['FLAG_HAUT_RISQUE'] = (
    (df_fe['NB_RETARDS_PAIEMENT_12M'] >= 2) |
    (df_fe['DEFAUT_ANTERIEUR'] == 1) |
    (df_fe['SCORE_CREDIT'] < 550) |
    (df_fe['RATIO_ENDETTEMENT'] > 0.6)
).astype('int8')

print("✓ FLAG_HAUT_RISQUE — flag composite des comportements passés à risque")


### 2.5 Variable d'interaction

In [ ]:
# --- FEATURE 6 : ENDETTEMENT_X_UTILISATION ---
# Produit du DTI (ratio dette/revenu) et du taux d'utilisation du crédit.
# Combine deux signaux complémentaires : charge globale de la dette + saturation du crédit.
# Leur produit amplifie les profils doublement contraints (endettés ET saturés).
# Un client avec DTI=0.5 et utilisation=0.9 ressort bien plus qu'avec chaque ratio seul.
df_fe['ENDETTEMENT_X_UTILISATION'] = df_fe['RATIO_ENDETTEMENT'] * df_fe['TAUX_UTILISATION_CREDIT']

print("✓ ENDETTEMENT_X_UTILISATION — interaction DTI × taux d'utilisation crédit")


### 2.6 Nettoyage et résumé

In [ ]:
# Remplacer les valeurs infinies par NaN
df_fe = df_fe.replace([np.inf, -np.inf], np.nan)

# Remplir les NaN créés par les divisions par la médiane
numeric_cols = df_fe.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df_fe[col].isnull().any():
        df_fe[col] = df_fe[col].fillna(df_fe[col].median())

# Résumé
new_cols = len(df_fe.columns) - initial_cols
engineered_features = [
    'CAPACITE_REMBOURSEMENT',    # capacité nette mensuelle
    'RATIO_PRET_REVENU_ANNUEL', # ratio prêt / revenu annuel
    'NB_TRANSACTIONS_TOTAL_30J',         # activité transactionnelle
    'TENDANCE_EPARGNE',         # tendance épargne 3 mois
    'FLAG_HAUT_RISQUE',        # flag risque historique
    'ENDETTEMENT_X_UTILISATION',       # interaction DTI x utilisation
]

print("=" * 50)
print("RÉSUMÉ DU FEATURE ENGINEERING")
print("=" * 50)
print(f"Colonnes initiales : {initial_cols}")
print(f"Nouvelles features : {new_cols}")
print(f"Colonnes finales   : {len(df_fe.columns)}")
print(f"\n{len(engineered_features)} features créées (vs 38 initiales) :")
for f in engineered_features:
    print(f"  - {f}")


In [ ]:
# Visualisation : Corrélation des nouvelles features avec TARGET
correlations = df_fe[engineered_features + ["TARGET"]].corr()["TARGET"].drop("TARGET").sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["red" if x < 0 else "green" for x in correlations.values]
correlations.plot(kind="barh", ax=ax, color=colors)
ax.set_xlabel("Corrélation avec TARGET")
ax.set_title("Corrélation des 6 nouvelles features avec la variable cible")
ax.axvline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.savefig(FIGURES_PATH / "correlation_nouvelles_features.png", dpi=150)
plt.show()


---

## 2.7 Sélection des variables (tests statistiques)

Sélection statistique des variables conformément à la méthodologie du cours de M. Bosco.

**Pipeline de sélection — 5 étapes dans l'ordre :**

| Étape | Variables testées | Test | Objectif |
|-------|------------------|------|----------|
| **1** | Qualitatives vs TARGET | **Chi² + V de Cramer** | Tester la dépendance de chaque var. qualitative avec la cible |
| **2** | Quantitatives ↔ Quantitatives | **Spearman** | Supprimer les variables trop corrélées entre elles (\|r\| > 0.80) |
| **3** | Quantitatives vs TARGET | **Mann-Whitney** | Tester si les distributions diffèrent entre défaut=0 et défaut=1 |
| **4** | Qualitatives ↔ Qualitatives | **Chi² + V de Cramer** | Supprimer les variables qualitatives redondantes entre elles (V > 0.70) |
| **5** | Quantitatives × Qualitatives | **Kruskal-Wallis** | Tester si une variable quant. varie selon les modalités d'une var. qual. |

In [ ]:
# ============================================================
# ÉTAPE 1 — Chi² + V de Cramer : variables qualitatives vs TARGET
# ============================================================
from scipy.stats import chi2_contingency
import numpy as np

print("=== ÉTAPE 1 : Chi² + V de Cramer — Variables qualitatives vs TARGET ===\n")

def cramers_v(table):
    """Calcule le V de Cramer à partir d'une table de contingence numpy."""
    chi2, p, dof, exp = chi2_contingency(table)
    n = table.sum()
    phi2 = chi2 / n
    r, k = table.shape
    phi2c = max(0, phi2 - (k-1)*(r-1)/(n-1))
    rc = r - (r-1)**2/(n-1)
    kc = k - (k-1)**2/(n-1)
    v = np.sqrt(phi2c / (min(kc-1, rc-1) + 1e-10))
    return v, p

# Identifier les colonnes qualitatives
EXCL = ['ID_CLIENT', 'TARGET', 'DATE_MOIS', 'DEFAUTS_ORIGINATION']
qual_cols = [c for c in df_fe.select_dtypes(include=['object','category']).columns
             if c not in EXCL]

P_SEUIL = 0.05
V_MIN   = 0.05   # effet minimum

chi2_target_results = []
for col in qual_cols:
    try:
        table = pd.crosstab(df_fe[col], df_fe['TARGET']).values
        if table.shape[0] < 2:
            continue
        v, p = cramers_v(table)
        chi2_target_results.append({
            'Variable'     : col,
            'n_modalites'  : table.shape[0],
            'p_value'      : round(p, 6),
            'V_Cramer'     : round(v, 4),
            'Discriminante': 'OUI' if p < P_SEUIL and v > V_MIN else 'NON'
        })
    except Exception as e:
        pass

df_etape1 = pd.DataFrame(chi2_target_results).sort_values('V_Cramer', ascending=False)
vars_qual_ok = df_etape1[df_etape1['Discriminante'] == 'OUI']['Variable'].tolist()
vars_qual_ko = df_etape1[df_etape1['Discriminante'] == 'NON']['Variable'].tolist()

print(f"Variables qualitatives analysées     : {len(qual_cols)}")
print(f"✓ Discriminantes (p<0.05, V>0.05)    : {len(vars_qual_ok)} → {vars_qual_ok}")
print(f"✗ Non discriminantes                 : {len(vars_qual_ko)}")
print()
print(df_etape1[['Variable','n_modalites','p_value','V_Cramer','Discriminante']].to_string(index=False))

In [ ]:
# ============================================================
# ÉTAPE 2 — Corrélation de Spearman : redondance entre variables quantitatives
# ============================================================
from itertools import combinations

print("=== ÉTAPE 2 : Spearman — Anti-redondance variables quantitatives ===\n")

EXCL_QUANT = ['ID_CLIENT', 'TARGET', 'DATE_MOIS', 'DEFAUTS_ORIGINATION']
quant_cols = [c for c in df_fe.select_dtypes(include='number').columns if c not in EXCL_QUANT]

R_SEUIL = 0.80
corr_matrix = df_fe[quant_cols].corr(method='spearman')

# Pour arbitrer entre 2 variables corrélées, on se base sur la corrélation Spearman avec TARGET
target_corr = {}
for col in quant_cols:
    target_corr[col] = abs(df_fe[col].corr(df_fe['TARGET'], method='spearman'))

vars_to_drop_spearman = set()
pairs_spearman = []

for col1, col2 in combinations(quant_cols, 2):
    r = corr_matrix.loc[col1, col2]
    if abs(r) > R_SEUIL:
        tc1 = target_corr.get(col1, 0)
        tc2 = target_corr.get(col2, 0)
        garder    = col1 if tc1 >= tc2 else col2
        supprimer = col2 if tc1 >= tc2 else col1
        vars_to_drop_spearman.add(supprimer)
        pairs_spearman.append({
            'Variable 1'   : col1, 'Variable 2': col2,
            'r Spearman'   : round(r, 3),
            'Corr TARGET V1': round(tc1, 4),
            'Corr TARGET V2': round(tc2, 4),
            'Gardée'       : garder,
            'Supprimée'    : supprimer
        })

df_etape2 = pd.DataFrame(pairs_spearman)
print(f"Variables quantitatives analysées    : {len(quant_cols)}")
print(f"Paires trop corrélées (|r| > {R_SEUIL})  : {len(df_etape2)}")
print(f"Variables supprimées (redondantes)   : {len(vars_to_drop_spearman)}")
if not df_etape2.empty:
    print()
    print(df_etape2.sort_values('r Spearman', ascending=False).to_string(index=False))

In [ ]:
# ============================================================
# ÉTAPE 3 — Mann-Whitney : variables quantitatives vs TARGET
# ============================================================
from scipy.stats import mannwhitneyu

print("=== ÉTAPE 3 : Mann-Whitney — Variables quantitatives vs TARGET ===\n")

# On teste uniquement les variables non supprimées à l'étape 2
quant_cols_restantes = [c for c in quant_cols if c not in vars_to_drop_spearman]

group0 = df_fe[df_fe['TARGET'] == 0]
group1 = df_fe[df_fe['TARGET'] == 1]

mw_results = []
for col in quant_cols_restantes:
    g0 = group0[col].dropna()
    g1 = group1[col].dropna()
    if len(g0) < 5 or len(g1) < 5:
        continue
    try:
        stat, p_val = mannwhitneyu(g0, g1, alternative='two-sided')
        from scipy.stats import norm
        z = norm.isf(p_val / 2) if p_val > 0 else 8
        r = abs(z) / np.sqrt(len(g0) + len(g1))
        mw_results.append({
            'Variable'     : col,
            'p_value'      : p_val,
            'effect_r'     : round(r, 4),
            'median_0'     : round(g0.median(), 3),
            'median_1'     : round(g1.median(), 3),
            'Discriminante': 'OUI' if p_val < 0.05 else 'NON'
        })
    except:
        pass

df_etape3 = pd.DataFrame(mw_results).sort_values('p_value')
vars_quant_ok = df_etape3[df_etape3['p_value'] < 0.05]['Variable'].tolist()
vars_quant_ko = df_etape3[df_etape3['p_value'] >= 0.05]['Variable'].tolist()

print(f"Variables quantitatives testées      : {len(quant_cols_restantes)}")
print(f"✓ Discriminantes (p < 0.05)          : {len(vars_quant_ok)}")
print(f"✗ Non discriminantes (p >= 0.05)     : {len(vars_quant_ko)}")
print()
print("TOP 20 variables les plus discriminantes :")
print(df_etape3.head(20)[['Variable','p_value','effect_r','median_0','median_1','Discriminante']].to_string(index=False))

In [ ]:
# ============================================================
# ÉTAPE 4 — Chi² + V de Cramer : redondance entre variables qualitatives
# ============================================================
from itertools import combinations

print("=== ÉTAPE 4 : Chi² + V de Cramer — Anti-redondance variables qualitatives ===\n")

# On teste uniquement les qualitatives discriminantes (étape 1)
V_INTER_SEUIL = 0.70
chi2_v_dict   = dict(zip(df_etape1['Variable'], df_etape1['V_Cramer']))

vars_to_drop_chi2 = set()
pairs_qual_dep = []

for col1, col2 in combinations(vars_qual_ok, 2):
    try:
        table = pd.crosstab(df_fe[col1], df_fe[col2]).values
        if table.shape[0] < 2 or table.shape[1] < 2:
            continue
        v_inter, _ = cramers_v(table)
        if v_inter > V_INTER_SEUIL:
            vt1 = chi2_v_dict.get(col1, 0)
            vt2 = chi2_v_dict.get(col2, 0)
            garder    = col1 if vt1 >= vt2 else col2
            supprimer = col2 if vt1 >= vt2 else col1
            vars_to_drop_chi2.add(supprimer)
            pairs_qual_dep.append({
                'Variable 1'     : col1, 'Variable 2': col2,
                'V Cramer inter' : round(v_inter, 4),
                'V vs TARGET V1' : round(vt1, 4),
                'V vs TARGET V2' : round(vt2, 4),
                'Gardée'         : garder,
                'Supprimée'      : supprimer
            })
    except:
        pass

df_etape4 = pd.DataFrame(pairs_qual_dep)
vars_qual_finales = [v for v in vars_qual_ok if v not in vars_to_drop_chi2]

print(f"Variables qualitatives testées       : {len(vars_qual_ok)}")
print(f"Paires trop dépendantes (V > {V_INTER_SEUIL})  : {len(df_etape4)}")
print(f"Variables supprimées (redondantes)   : {len(vars_to_drop_chi2)}")
if not df_etape4.empty:
    print()
    print(df_etape4.to_string(index=False))
print(f"\n✓ Variables qualitatives conservées : {vars_qual_finales}")

In [ ]:
# ============================================================
# ETAPE 5 - Kruskal-Wallis : croisement variables quantitatives x qualitatives
# ============================================================
from scipy.stats import kruskal

print("=== ETAPE 5 : Kruskal-Wallis - Croisement Quantitatives x Qualitatives ===")
print(f"vars_quant_ok     ({len(vars_quant_ok)}) : {vars_quant_ok[:5]}...")
print(f"vars_qual_finales ({len(vars_qual_finales)}) : {vars_qual_finales}")
print()

kw_results = []
for q_col in vars_quant_ok:
    for cat_col in vars_qual_finales:
        groupes = [df_fe.loc[df_fe[cat_col] == mod, q_col].dropna().values
                   for mod in df_fe[cat_col].unique()]
        groupes = [g for g in groupes if len(g) >= 5]
        if len(groupes) < 2:
            continue
        try:
            stat, p_val = kruskal(*groupes)
            kw_results.append({
                "Var_quant" : q_col,
                "Var_qual"  : cat_col,
                "H_stat"    : round(stat, 3),
                "p_value"   : round(p_val, 6),
                "Lien_sig"  : "OUI" if p_val < 0.05 else "NON"
            })
        except Exception:
            pass  # groupes identiques ou variance nulle

df_etape5 = pd.DataFrame(kw_results if kw_results else [],
                         columns=["Var_quant","Var_qual","H_stat","p_value","Lien_sig"])
if kw_results:
    df_etape5 = df_etape5.sort_values("p_value")

# n_sig toujours defini, meme si aucun resultat
n_sig = int((df_etape5["p_value"] < 0.05).sum()) if not df_etape5.empty else 0

if df_etape5.empty:
    print("Aucune paire (quant x qual) valide — etape 5 ignoree.")
    if not vars_qual_finales:
        print("  -> Aucune variable qualitative discriminante apres etapes 1/4.")
else:
    print(f"Paires testees (quant x qual)  : {len(df_etape5)}")
    print(f"Liens significatifs (p < 0.05) : {n_sig}")
    print()
    print(df_etape5[["Var_quant","Var_qual","H_stat","p_value","Lien_sig"]].to_string(index=False))


In [ ]:
# ============================================================
# BILAN FINAL - Construction du dataset selectionne
# ============================================================
ENGINEERED = ["CAPACITE_REMBOURSEMENT", "RATIO_PRET_REVENU_ANNUEL", "NB_TRANSACTIONS_TOTAL_30J",
              "TENDANCE_EPARGNE", "FLAG_HAUT_RISQUE", "ENDETTEMENT_X_UTILISATION"]

vars_finales = vars_quant_ok + vars_qual_finales

# --- Force inclusion des features construites (domain knowledge) ---
# Elles ont ete construites specifiquement pour le scoring.
# Meme si Spearman les a eliminees (car correlees aux variables d'origine),
# elles apportent une information combinee que les originales n'ont pas.
added = []
for f in ENGINEERED:
    if f not in vars_finales and f in df_fe.columns:
        vars_finales.append(f)
        added.append(f)

cols_finales = ["ID_CLIENT","TARGET","DATE_MOIS","DEFAUTS_ORIGINATION"] + vars_finales
cols_finales = [c for c in cols_finales if c in df_fe.columns]
df_selected  = df_fe[cols_finales].copy()

print("=" * 60)
print("BILAN FINAL - SELECTION DES VARIABLES")
print("=" * 60)
print(f"  Variables avant selection              : {df_fe.shape[1]}")
print(f"  Etape 1 - Qual. non discriminantes     : {len(vars_qual_ko)}")
print(f"  Etape 2 - Quant. supprimees (Spearman) : {len(vars_to_drop_spearman)}")
print(f"  Etape 3 - Quant. non discriminantes    : {len(vars_quant_ko)}")
print(f"  Etape 4 - Qual. redondantes (Chi2)     : {len(vars_to_drop_chi2)}")
print(f"  Etape 5 - Liens quantxqual identifies  : {n_sig} paires significatives")
print()

# --- Diagnostic : sort des 6 features construites ---
print("--- SUIVI DES 6 FEATURES CONSTRUITES ---")
for f in ENGINEERED:
    if f in vars_quant_ok:
        print(f"  {f:30s} -> RETENUE par selection statistique")
    elif f in added:
        print(f"  {f:30s} -> FORCEE (eliminee par Spearman, rajoutee par domain knowledge)")
    else:
        print(f"  {f:30s} -> RETENUE")
print()

print(f"  Variables finales retenues : {len(vars_finales)}")
print(f"       dont quantitatives   : {sum(1 for v in vars_finales if v in df_fe.select_dtypes(include="number").columns)}")
print(f"       dont qualitatives    : {sum(1 for v in vars_finales if v in df_fe.select_dtypes(include=["object","category"]).columns)}")
print(f"       dont construites     : {len(ENGINEERED)}")
print()
print(f"  Liste complete : {sorted(vars_finales)}")
print(f"\ndf_selected : {df_selected.shape[0]:,} lignes x {df_selected.shape[1]} colonnes")


### 3.2 Standardisation

In [ ]:
# ============================================================
# 3.1 — Séparation Train / Test
# ============================================================
from sklearn.model_selection import train_test_split

feature_cols = [c for c in df_selected.columns
                if c not in ['ID_CLIENT', 'TARGET', 'DATE_MOIS', 'DEFAUTS_ORIGINATION']]

X = df_selected[feature_cols]
y = df_selected['TARGET']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Train : {X_train.shape[0]:,} observations")
print(f"Test  : {X_test.shape[0]:,} observations")
print(f"Ratio défauts train : {y_train.mean():.2%}")
print(f"Ratio défauts test  : {y_test.mean():.2%}")

In [ ]:
# Encodage des variables catégorielles (Label Encoding)
from sklearn.preprocessing import LabelEncoder

cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Colonnes catégorielles à encoder ({len(cat_cols)}) : {cat_cols}")

le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].astype(str))
    X_test[col]  = le.transform(X_test[col].astype(str))
    le_dict[col] = le

print("✓ Encodage effectué")

In [ ]:
# Standardisation des features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✓ Standardisation effectuée")
print(f"  Train : moyenne ≈ 0, écart-type ≈ 1")

### 3.3 Rééquilibrage avec SMOTE

In [ ]:
# Application de SMOTE pour gérer le déséquilibre des classes
print("Application de SMOTE...")
print(f"\nAvant SMOTE :")
print(f"  - Total : {len(y_train):,}")
print(f"  - Classe 0 : {(y_train == 0).sum():,}")
print(f"  - Classe 1 : {(y_train == 1).sum():,}")

from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=3)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print(f"\nAprès SMOTE :")
print(f"  - Total : {len(y_train_resampled):,}")
print(f"  - Classe 0 : {(y_train_resampled == 0).sum():,}")
print(f"  - Classe 1 : {(y_train_resampled == 1).sum():,}")

---

## 4. Modélisation : Régression Logistique (Baseline)

On commence par un modèle simple et interprétable pour établir une baseline.

In [ ]:
# Entraînement du modèle
print("Entraînement de la régression logistique...")

model = LogisticRegression(
    C=1.0,
    class_weight='balanced',
    max_iter=1000,
    random_state=RANDOM_STATE,
    solver='lbfgs',
    n_jobs=-1
)

model.fit(X_train_resampled, y_train_resampled)

print("✓ Modèle entraîné")

In [ ]:
# Prédictions sur le jeu de test
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

print(f"Prédictions effectuées sur {len(y_test):,} observations")

### 4.1 Métriques de performance

In [ ]:
# Calcul des métriques
metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1-Score': f1_score(y_test, y_pred),
    'AUC-ROC': roc_auc_score(y_test, y_pred_proba)
}

print("=" * 50)
print("RÉSULTATS SUR LE JEU DE TEST")
print("=" * 50)
for metric, value in metrics.items():
    highlight = "⭐" if metric == 'AUC-ROC' else " "
    print(f"{highlight} {metric:12} : {value:.4f}")
print("=" * 50)

In [ ]:
# Rapport de classification détaillé
print("\nRAPPORT DE CLASSIFICATION")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=['Non défaut', 'Défaut']))

In [ ]:
# Validation croisée sur le jeu d'entraînement
print("Validation croisée (5-fold) sur le train...")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)

print(f"\nAUC-ROC CV : {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")
print(f"Scores par fold : {[f'{s:.4f}' for s in cv_scores]}")

### 4.4 Coefficient de Gini

Le **coefficient de Gini** est la métrique principale en scoring de crédit.  
Il mesure le pouvoir discriminant du modèle entre les bons et mauvais payeurs.

> **Formule** : Gini = 2 × AUC − 1

| Interprétation | Gini |
|----------------|------|
| Modèle parfait | 1.00 |
| Bon modèle     | > 0.60 |
| Modèle acceptable | 0.40 – 0.60 |
| Modèle faible  | < 0.40 |
| Aléatoire      | 0.00 |

On calcule le Gini sur le **train** et le **test** pour détecter un éventuel surapprentissage.

In [ ]:
# ============================================================
# 4.4 — Coefficient de Gini (Train vs Test)
# ============================================================
from sklearn.metrics import roc_auc_score

print("=" * 50)
print("COEFFICIENT DE GINI")
print("=" * 50)

# ── Gini sur le jeu de TEST ──
auc_test  = roc_auc_score(y_test, y_pred_proba)
gini_test = 2 * auc_test - 1

# ── Gini sur le jeu de TRAIN ──
y_train_proba = model.predict_proba(X_train_scaled)
auc_train  = roc_auc_score(y_train, y_train_proba[:, 1])
gini_train = 2 * auc_train - 1

print(f"  AUC  Train : {auc_train:.4f}   →   Gini Train : {gini_train:.4f}")
print(f"  AUC  Test  : {auc_test:.4f}   →   Gini Test  : {gini_test:.4f}")
print()

# ── Diagnostic surapprentissage ──
ecart = gini_train - gini_test
print(f"  Écart Train - Test : {ecart:.4f}")
if ecart > 0.15:
    print("  ⚠️  Surapprentissage détecté (écart > 0.15) — envisager régularisation")
elif ecart > 0.05:
    print("  ⚡ Léger surapprentissage (écart > 0.05) — à surveiller")
else:
    print("  ✓  Bonne généralisation (écart faible)")

# ── Interprétation du Gini test ──
print()
if gini_test >= 0.60:
    print(f"  ✓  Gini test = {gini_test:.4f} → Bon modèle")
elif gini_test >= 0.40:
    print(f"  ⚡ Gini test = {gini_test:.4f} → Modèle acceptable")
else:
    print(f"  ✗  Gini test = {gini_test:.4f} → Modèle faible — amélioration nécessaire")

# ── Visualisation ──
fig, ax = plt.subplots(figsize=(6, 4))
barres = ax.bar(['Gini Train', 'Gini Test'], [gini_train, gini_test],
                color=['#3498db', '#e74c3c'], width=0.4, edgecolor='black')
ax.axhline(y=0.60, color='green', linestyle='--', linewidth=1.5, label='Seuil bon modèle (0.60)')
ax.axhline(y=0.40, color='orange', linestyle='--', linewidth=1.5, label='Seuil acceptable (0.40)')
for barre in barres:
    ax.text(barre.get_x() + barre.get_width()/2, barre.get_height() + 0.01,
            f'{barre.get_height():.4f}', ha='center', va='bottom', fontweight='bold')
ax.set_ylim(0, 1)
ax.set_ylabel('Coefficient de Gini')
ax.set_title('Gini Train vs Test — Régression Logistique', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(ROOT_DIR / 'outputs' / 'figures' / '02_modeling' / 'gini_train_test.png', bbox_inches='tight')
plt.show()
print("Figure sauvegardée : outputs/figures/02_modeling/gini_train_test.png")

### 4.2 Visualisations des performances

In [ ]:
# Figure avec 4 graphiques
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Courbe ROC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
axes[0, 0].plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {metrics["AUC-ROC"]:.3f})')
axes[0, 0].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Aléatoire')
axes[0, 0].fill_between(fpr, tpr, alpha=0.3)
axes[0, 0].set_xlabel('Taux de Faux Positifs (FPR)')
axes[0, 0].set_ylabel('Taux de Vrais Positifs (TPR)')
axes[0, 0].set_title('Courbe ROC')
axes[0, 0].legend(loc='lower right')
axes[0, 0].grid(True, alpha=0.3)

# 2. Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 1],
            xticklabels=['Non défaut', 'Défaut'],
            yticklabels=['Non défaut', 'Défaut'])
axes[0, 1].set_xlabel('Classe prédite')
axes[0, 1].set_ylabel('Classe réelle')
axes[0, 1].set_title('Matrice de Confusion')

# 3. Distribution des probabilités prédites
axes[1, 0].hist(y_pred_proba[y_test == 0], bins=50, alpha=0.7, label='Non défaut', color='green', density=True)
axes[1, 0].hist(y_pred_proba[y_test == 1], bins=50, alpha=0.7, label='Défaut', color='red', density=True)
axes[1, 0].axvline(0.5, color='black', linestyle='--', linewidth=2, label='Seuil = 0.5')
axes[1, 0].set_xlabel('Probabilité de défaut prédite')
axes[1, 0].set_ylabel('Densité')
axes[1, 0].set_title('Distribution des probabilités prédites')
axes[1, 0].legend()

# 4. Courbe Precision-Recall
precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
axes[1, 1].plot(recall, precision, 'b-', linewidth=2)
axes[1, 1].fill_between(recall, precision, alpha=0.3)
axes[1, 1].set_xlabel('Recall')
axes[1, 1].set_ylabel('Precision')
axes[1, 1].set_title('Courbe Precision-Recall')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_PATH / 'performance_modele_baseline.png', dpi=150)
plt.show()

### 4.3 Importance des features

In [ ]:
# Importance des features (valeur absolue des coefficients)
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': model.coef_[0],
    'importance': np.abs(model.coef_[0])
}).sort_values('importance', ascending=False)

print("TOP 15 FEATURES LES PLUS IMPORTANTES")
print("=" * 50)
for i, row in feature_importance.head(15).iterrows():
    sign = "+" if row['coefficient'] > 0 else "-"
    print(f"{sign} {row['feature']:35} : {row['importance']:.4f}")

In [ ]:
# Visualisation importance des features
n_feat = min(len(feature_importance), 25)
top_n = feature_importance.head(n_feat).sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(10, max(4, n_feat * 0.4)))
colors = ["red" if c < 0 else "green" for c in top_n["coefficient"]]
ax.barh(top_n["feature"], top_n["importance"], color=colors, alpha=0.8)
ax.set_xlabel("Importance (|coefficient|)")
ax.set_title(f"Top {n_feat} Features - Regression Logistique\n"
             f"(Vert = risque de defaut, Rouge = protection)")
plt.tight_layout()
plt.savefig(FIGURES_PATH / "importance_features_baseline.png", dpi=150)
plt.show()


---

## 5. Analyse temporelle préliminaire

Étude de la série `DEFAUTS_ORIGINATION` pour préparer la modélisation SARIMA en Phase 2.

In [ ]:
# Agrégation mensuelle des défauts
monthly_data = df_fe.groupby('DATE_MOIS').agg({
    'TARGET': ['count', 'sum', 'mean'],
    'DEFAUTS_ORIGINATION': 'first'
}).reset_index()

monthly_data.columns = ['DATE_MOIS', 'nb_dossiers', 'nb_defauts', 'taux_defaut', 'defaults_origination']
monthly_data['taux_defaut'] *= 100

print(f"Période : {monthly_data['DATE_MOIS'].min()} à {monthly_data['DATE_MOIS'].max()}")
print(f"Nombre de mois : {len(monthly_data)}")

In [ ]:
# Statistiques descriptives de DEFAUTS_ORIGINATION
print("\nStatistiques DEFAUTS_ORIGINATION :")
print(f"  Moyenne     : {monthly_data['defaults_origination'].mean():.1f}")
print(f"  Écart-type  : {monthly_data['defaults_origination'].std():.1f}")
print(f"  Minimum     : {monthly_data['defaults_origination'].min()}")
print(f"  Maximum     : {monthly_data['defaults_origination'].max()}")
print(f"  Médiane     : {monthly_data['defaults_origination'].median():.1f}")

In [ ]:
# Visualisation de la série temporelle
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# 1. Volume de défauts par mois
axes[0].bar(monthly_data['DATE_MOIS'], monthly_data['defaults_origination'], 
            color='coral', alpha=0.8, width=20)
axes[0].axhline(monthly_data['defaults_origination'].mean(), color='red', 
                linestyle='--', label=f"Moyenne = {monthly_data['defaults_origination'].mean():.0f}")
axes[0].set_ylabel('Nombre de défauts')
axes[0].set_title('Volume de défauts par mois (DEFAUTS_ORIGINATION)')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)

# 2. Taux de défaut mensuel
axes[1].plot(monthly_data['DATE_MOIS'], monthly_data['taux_defaut'], 
             marker='o', color='steelblue', linewidth=2, markersize=4)
axes[1].axhline(monthly_data['taux_defaut'].mean(), color='red', 
                linestyle='--', label=f"Moyenne = {monthly_data['taux_defaut'].mean():.2f}%")
axes[1].set_ylabel('Taux de défaut (%)')
axes[1].set_title('Évolution du taux de défaut mensuel')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=45)

# 3. Volume de dossiers par mois
axes[2].bar(monthly_data['DATE_MOIS'], monthly_data['nb_dossiers'], 
            color='teal', alpha=0.7, width=20)
axes[2].set_ylabel('Nombre de dossiers')
axes[2].set_xlabel('Date')
axes[2].set_title('Volume de dossiers par mois')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(FIGURES_PATH / 'serie_temporelle_defauts.png', dpi=150)
plt.show()

In [ ]:
# Analyse de tendance
monthly_data['month_num'] = range(len(monthly_data))
correlation = monthly_data['month_num'].corr(monthly_data['defaults_origination'])

if correlation > 0.3:
    trend = "CROISSANTE 📈"
elif correlation < -0.3:
    trend = "DÉCROISSANTE 📉"
else:
    trend = "STABLE ➡️"

print("\nANALYSE DE TENDANCE")
print("=" * 40)
print(f"Corrélation temps/défauts : {correlation:.3f}")
print(f"Tendance : {trend}")

In [ ]:
# Analyse de saisonnalité
monthly_data['month_of_year'] = pd.to_datetime(monthly_data['DATE_MOIS']).dt.month
seasonality = monthly_data.groupby('month_of_year')['defaults_origination'].mean()

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(seasonality.index, seasonality.values, color='teal', alpha=0.8)
ax.axhline(seasonality.mean(), color='red', linestyle='--', 
           label=f'Moyenne = {seasonality.mean():.0f}')

# Colorer les mois au-dessus de la moyenne
for i, bar in enumerate(bars):
    if seasonality.values[i] > seasonality.mean():
        bar.set_color('coral')

ax.set_xlabel('Mois de l\'année')
ax.set_ylabel('Moyenne des défauts')
ax.set_title('Saisonnalité des défauts (moyenne par mois)')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan', 'Fév', 'Mar', 'Avr', 'Mai', 'Jun',
                    'Jul', 'Aoû', 'Sep', 'Oct', 'Nov', 'Déc'])
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'saisonnalite_defauts.png', dpi=150)
plt.show()

print("\nMois avec le plus de défauts :")
top_months = seasonality.sort_values(ascending=False).head(3)
for month, value in top_months.items():
    month_name = ['Jan', 'Fév', 'Mar', 'Avr', 'Mai', 'Jun',
                  'Jul', 'Aoû', 'Sep', 'Oct', 'Nov', 'Déc'][month-1]
    print(f"  {month_name} : {value:.0f} défauts en moyenne")

---

## 6. Sauvegarde des résultats

In [ ]:
# Sauvegarder les données avec les nouvelles features
features_path = ROOT_DIR / 'data' / 'processed' / 'features_engineered.parquet'
df_fe.to_parquet(features_path, index=False)
print(f"✓ Données avec features : {features_path}")

# Sauvegarder le modèle
model_path = MODELS_PATH / 'logistic_regression_baseline.joblib'
joblib.dump({
    'model': model,
    'scaler': scaler,
    'feature_cols': feature_cols,
    'label_encoders': le_dict
}, model_path)
print(f"✓ Modèle sauvegardé : {model_path}")

# Sauvegarder les métriques
import json
metrics_path = ROOT_DIR / 'outputs' / 'reports' / '03_modeling' / 'metriques_modele_baseline.json'
with open(metrics_path, 'w') as f:
    json.dump({
        'model': 'LogisticRegression',
        'metrics': {k: float(v) for k, v in metrics.items()},
        'cv_auc_mean': float(cv_scores.mean()),
        'cv_auc_std': float(cv_scores.std()),
        'timestamp': datetime.now().isoformat()
    }, f, indent=2)
print(f"✓ Métriques : {metrics_path}")

# Sauvegarder l'importance des features
importance_path = ROOT_DIR / 'outputs' / 'reports' / '03_modeling' / 'importance_variables.csv'
feature_importance.to_csv(importance_path, index=False)
print(f"✓ Importance des features : {importance_path}")

---

## 7. Résumé et conclusions

### Feature Engineering
- **6 features ciblées** créées (vs 38 initialement)
- Chacune capture un aspect distinct du risque : capacité de remboursement, proportionnalité du prêt, activité transactionnelle, tendance épargne, historique de risque, double contrainte DTI×utilisation

### Modèle Baseline (Régression Logistique)
- AUC-ROC et Gini calculés dynamiquement (voir cellules ci-dessus)
- Validation croisée 5-fold
- SMOTE (k_neighbors=3) pour gérer le déséquilibre des classes

### Analyse Temporelle
- **Tendance** : Croissante
- **Saisonnalité** : Pics en fin d'année
- Prêt pour modélisation SARIMA en Phase 2

### Prochaines étapes (Phase 2)
- [ ] Optimisation des hyperparamètres
- [ ] Modèles avancés : Random Forest, XGBoost
- [ ] Modèle SARIMA pour séries temporelles
- [ ] Prévisions à 3 mois


In [ ]:
print("=" * 60)
print("PHASE 1 - PARTIE 2 TERMINÉE")
print("=" * 60)
print(f"\nDate : {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"\nFichiers générés :")
print(f"  - data/processed/features_engineered.parquet")
print(f"  - outputs/models/logistic_regression_baseline.joblib")
print(f"  - outputs/figures/02_modeling/*.png")
print(f"\nAUC-ROC du modèle baseline : {metrics['AUC-ROC']:.4f}")